In [1]:
# imports
# Los secretos (ANTHROPIC_API_KEY, NEO4J__PASSWORD) se leen de .env al importar src.config.
# Flujo: .env → load_dotenv() en config.py → Settings() → settings.anthropic_api_key / settings.neo4j.password
# Todos los módulos de src/ reciben los secretos a través del singleton `settings`.

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.config import settings
from src.api import BOEDownloader
from src.preprocess import Preprocesador, generar_esquemas
from src.llm import Llm

print("✓ Imports OK")
print(f"  Neo4j URI:  {settings.neo4j.uri}")
print(f"  LLM model:  {settings.llm.model}")
print(f"  Max exchanges: {settings.llm.max_exchanges}")

✓ Imports OK
  Neo4j URI:  bolt://localhost:7687
  LLM model:  claude-haiku-4-5
  Max exchanges: 1


In [2]:
# API

api = BOEDownloader()
resumen = api.descargar_masivo()
resumen

2026-06-02 19:35:14 [info     ] 
Obteniendo ids...            


2026-06-02 19:35:28 [info     ] Listado pagina                 obtenidos=10000 offset=0
2026-06-02 19:35:31 [info     ] Listado pagina                 obtenidos=2286 offset=10000
2026-06-02 19:35:31 [info     ] 
Ids persistidos               path=/home/jorgee/reversa/ontology/kinetic-layer/api_boe/ids.txt total=12286
2026-06-02 19:35:31 [info     ] 
Procesando ids...            


100%|██████████| 12286/12286 [14:11<00:00, 14.42norma/s, BOE-A-2026-11627] 

2026-06-02 19:49:43 [info     ] 
Descarga masiva completada    descargados=12286 fallidos=0 saltados=0 total=12286
2026-06-02 19:49:43 [info     ] 
Reintento completado          recuperados=0 total=0


ResumenDescarga(total=12286, descargados=12286, fallidos=0, saltados=0)

In [3]:
# Preprocesar — parsear XMLs y cargar en Neo4j

preprocesador = Preprocesador()
resumen = preprocesador.preprocesar_todo()
resumen

2026-06-02 19:49:43 [info     ] Grafo limpiado                
2026-06-02 19:49:43 [info     ] 
Preprocesando...             


100%|██████████| 12286/12286 [09:41<00:00, 21.14norma/s, BOE-A-2026-9961]  

2026-06-02 19:59:24 [info     ] 
Reintento completado          recuperados=0 total=0
2026-06-02 19:59:24 [info     ] Faltantes escritos             anteriores=10081 posteriores=7622
2026-06-02 19:59:24 [info     ] 
Esquemas creados              semantic_dir=/home/jorgee/reversa/ontology/semantic-layer
2026-06-02 19:59:24 [info     ] 
Preprocesado completado       aristas=49725 errores=0 nodos=12286 procesadas=12286


ResumenPreproc(procesadas=12286, nodos_upsert=12286, aristas_upsert=49725, errores=0)

In [4]:
def borrar_nodos_dinamicos_bbdd() -> dict:
    """Elimina todos los nodos :UserQuery y aristas :RESULT_EDGE del grafo.

    Útil para limpiar el grafo dinámico entre sesiones de prueba.

    Returns:
        Resumen con el número de nodos y aristas eliminados.
    """
    from neo4j import GraphDatabase

    driver = GraphDatabase.driver(
        settings.neo4j.uri,
        auth=(settings.neo4j.user, settings.neo4j.password),
    )
    with driver.session(database=settings.neo4j.database) as session:
        result = session.run("MATCH (q:UserQuery) DETACH DELETE q RETURN count(q) AS eliminados")
        eliminados = result.single()["eliminados"]
    driver.close()
    return {"nodos_eliminados": eliminados}


print(borrar_nodos_dinamicos_bbdd())

{'nodos_eliminados': 0}


In [5]:
# Neo4j — stats rápidas del grafo
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    settings.neo4j.uri,
    auth=(settings.neo4j.user, settings.neo4j.password),
)
with driver.session(database=settings.neo4j.database) as session:

    def q(cypher):
        return session.run(cypher).single()["c"]

    stats = {
        "total_normas": q("MATCH (n:Norma) RETURN count(n) AS c"),
        "normas_consolidadas": q("MATCH (n:Norma) WHERE size(keys(n)) > 1 RETURN count(n) AS c"),
        "no_consolidadas_origen": q(
            "MATCH (a:Norma)-[]->(b:Norma) WHERE size(keys(a)) = 1 AND size(keys(b)) > 1 RETURN count(DISTINCT a) AS c"
        ),
        "no_consolidadas_destino": q(
            "MATCH (a:Norma)-[]->(b:Norma) WHERE size(keys(a)) > 1 AND size(keys(b)) = 1 RETURN count(DISTINCT b) AS c"
        ),
        "no_consolidadas_solapadas": q(
            "MATCH (a:Norma)-[]->(x:Norma)-[]->(b:Norma) WHERE size(keys(x)) = 1 AND size(keys(a)) > 1 AND size(keys(b)) > 1 RETURN count(DISTINCT x) AS c"
        ),
        "vigentes": q("MATCH (n:Norma {vigente: true}) RETURN count(n) AS c"),
        "total_aristas": q("MATCH ()-[r]->() RETURN count(r) AS c"),
        "aristas_CITA": q("MATCH ()-[:CITA]->() RETURN count(*) AS c"),
        "aristas_DEROGA": q("MATCH ()-[:DEROGA]->() RETURN count(*) AS c"),
        "aristas_MODIFICA": q("MATCH ()-[:MODIFICA]->() RETURN count(*) AS c"),
        "user_queries": q("MATCH (q:UserQuery) RETURN count(q) AS c"),
    }
driver.close()

print("Stats del grafo:")
for k, v in stats.items():
    print(f"  {k:<30} {v:>8,}")

Stats del grafo:
  total_normas                     29,014
  normas_consolidadas              12,286
  no_consolidadas_origen            7,622
  no_consolidadas_destino          10,081
  no_consolidadas_solapadas           975
  vigentes                          9,910
  total_aristas                    49,725
  aristas_CITA                     14,117
  aristas_DEROGA                   14,248
  aristas_MODIFICA                 21,360
  user_queries                          0


In [6]:
# Generar esquemas semánticos

generar_esquemas()

2026-06-02 19:59:25 [info     ] 
Esquemas creados              semantic_dir=/home/jorgee/reversa/ontology/semantic-layer


In [7]:
# LLM — prueba de una sola pregunta
async def test_llm_simple(pregunta: str = "¿Cuántas normas vigentes hay en el grafo?"):
    llm = Llm()
    print(f"Pregunta: {pregunta}\n")
    print("Respuesta: ", end="", flush=True)
    async for token in llm.responder(pregunta):
        print(token, end="", flush=True)
    print()
    llm.close()


await test_llm_simple()

Pregunta: ¿Cuántas normas vigentes hay en el grafo?

Respuesta: 

El grafo contiene **9.910 normas vigentes** actualmente.


## Briefing

##### 1 Diagnosis: which laws have become unreadable? (The 5 norms most amended by other norms.)

In [8]:
pregunta = """
The Vice-Presidency wants the consolidation backlog: the laws amended so many times they are now incomprehensible even to lawyers. Give the top 5. These are the first candidates for a clean rewrite.
"""
await test_llm_simple(pregunta)

Pregunta: 
The Vice-Presidency wants the consolidation backlog: the laws amended so many times they are now incomprehensible even to lawyers. Give the top 5. These are the first candidates for a clean rewrite.


Respuesta: 

Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `MODIFICADA_POR` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=51, offset=50>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 50, 'line': 1, 'column': 51}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (ley:Norma {rango: 'Ley', vigente: true})-[:MODIFICADA_POR]-(mod) RETURN ley.id, ley.numero_oficial, count(mod) as num_modificaciones ORDER BY num_modificaciones DESC LIMIT 5"


**Top 5 leyes con mayor backlog de consolidación:**

1. [BOE-A-1988-29622 — 38/1988] - 133 modificaciones
2. [BOE-A-1992-28740 — 37/1992] - 79 modificaciones
3. [BOE-A-2006-20764 — 35/2006] - 74 modificaciones
4. [BOE-A-2000-323 — 1/2000] - 69 modificaciones
5. [BOE-A-1992-28741 — 38/1992] - 56 modificaciones

Estos son los candidatos prioritarios para reescritura consolidada.


##### 2 Root cause: who made the mess? (The 5 norms that amend the most other norms.)

In [9]:
pregunta = """
Show the Council how laws become unreadable. Find the “omnibus” laws — single acts that silently rewrote dozens of unrelated statutes at once. Name the top 5 worst offenders. The Council wants to see the pattern, not just the symptom.
"""
await test_llm_simple(pregunta)

Pregunta: 
Show the Council how laws become unreadable. Find the “omnibus” laws — single acts that silently rewrote dozens of unrelated statutes at once. Name the top 5 worst offenders. The Council wants to see the pattern, not just the symptom.


Respuesta: **The Pattern: How Laws Become Unreadable**

The Council should understand this is not a bug—it's a structural feature of Spanish legislative practice. Here are the **top 5 omnibus offenders**:

**1. BOE-A-2003-23936 — Ley 62/2003** (68 normas modificadas)
Titled "Medidas fiscales, administrativas y del orden social" — a single law that silently rewrote the Commercial Code (1885), the Mortgage Law (1946), Civil Service Law (1964), Nuclear Energy Law (1964), Maritime Social Security (1974), Metallic Currency Law (1975), National Security Council Law (1980), National Parks Law (1981), Disability Integration Law (1982), Public Administration Reform (1984), Financial Intermediaries Law (1985), Historical Heritage Law (1985), Savings Ba

##### 3 The rot: how much of the statute book rests on dead law? (The percentage of in-force norms that cite at least one repealed norm, and the 5 repealed norms most cited by in-force norms.)

In [10]:
pregunta = """
Find every law still in force that cites a law already repealed. Quantify it: what fraction of live Spanish law rests on legal ground that no longer exists? Then surface the top 5 most-cited ghosts — the dead laws still propping up the most live statutes.
"""
await test_llm_simple(pregunta)

Pregunta: 
Find every law still in force that cites a law already repealed. Quantify it: what fraction of live Spanish law rests on legal ground that no longer exists? Then surface the top 5 most-cited ghosts — the dead laws still propping up the most live statutes.


Respuesta: **Summary:**

**Fraction of live Spanish law resting on repealed legal ground:**
771 out of 9,910 vigentes normas cite repealed laws = **7.78%** of Spanish law in force still references laws that have been derogated.

**Top 5 most-cited "ghost laws" propping up live statutes:**

1. **BOE-A-1992-26318** — Ley 30/1992, de 26 de noviembre, de Régimen Jurídico de las Administraciones Públicas y del Procedimiento Administrativo Común (cited by 240 vigentes normas)

2. **BOE-A-1988-18764** — Ley 24/1988, de 28 de julio, del Mercado de Valores (cited by 35 vigentes normas)

3. **BOE-A-2007-12352** — Ley 11/2007, de 22 de junio, de acceso electrónico de los ciudadanos a los Servicios Públicos (cited by 32 vigentes norm

##### 4 The scalpel: the unfinished repeal. (The list of in-force norms that cite Ley 30/1992 directly.)

In [11]:
pregunta = """
In 2015 the Council repealed Ley 30/1992, the act on the Legal Regime of Public Administrations and Common Administrative Procedure, and replaced it with Leyes 39/2015 and 40/2015. But the cleanup was never finished: dozens of laws still in force keep citing Ley 30/1992 as if it existed. The Council wants to close the operation — update those orphan references — and needs the worklist. Compute the blast radius of Ley 30/1992: the laws still in force that cite it directly.
"""
await test_llm_simple(pregunta)

Pregunta: 
In 2015 the Council repealed Ley 30/1992, the act on the Legal Regime of Public Administrations and Common Administrative Procedure, and replaced it with Leyes 39/2015 and 40/2015. But the cleanup was never finished: dozens of laws still in force keep citing Ley 30/1992 as if it existed. The Council wants to close the operation — update those orphan references — and needs the worklist. Compute the blast radius of Ley 30/1992: the laws still in force that cite it directly.


Respuesta: The blast radius of Ley 30/1992 is substantial. There are **176 laws still in force** that cite it directly. Here's the complete worklist:

[BOE-A-1993-13321 — 630/1993], [BOE-A-1993-13435 — 677/1993], [BOE-A-1993-13669 — 682/1993], [BOE-A-1993-13670 — 683/1993], [BOE-A-1993-13993 — Ley Foral 2/1993], [BOE-A-1993-17028 — Circular 9/6/1993], [BOE-A-1993-17031 — 853/1993], [BOE-A-1993-21947 — 8/1993], [BOE-A-1993-23229 — Orden 10/9/1993], [BOE-A-1993-27920 — 8/1993], [BOE-A-1993-29257 — 3/1993], 